# 25 — AssayGroups

Group multiple sensor channels into a named `AssayGroup` and reuse that group across studies and comparisons.

**Dataset**: XJTU-SY — Bearing 1_1 and siblings  
**API**: `AssayGroup` · `study.assay_group()` · `study.lifecycle_features(group=...)` · `group.compare_with()`

In [1]:
import warnings, logging, sys
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

WRAPPER_ROOT = Path("..").resolve()
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper, AssayGroup
from isa_phm.plotter import ISAPlotter
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

ISA_JSON = Path(r"G:\ISA\Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM-Out.json")
print("Exists:", ISA_JSON.exists())

Loading BokehJS ...

Exists: True


In [2]:
wrapper = ISAWrapper(ISA_JSON, strict_validation=False, cache_maxsize=20)
plotter = ISAPlotter()

study = wrapper.study("Bearing 1_1")

print(f"Study  : {study.title}")
print(f"Sensors: {[a.assay_id for a in study.list_assays()]}")

Study  : Bearing 1_1
Sensors: ['a_st01_se01', 'a_st01_se02']


## 1. Define an AssayGroup

`AssayGroup` is a **study-agnostic** descriptor — it holds a list of assay IDs (or 1-based integer indices) and a name.  
Define it once and bind it to any study with `study.assay_group(group)`.

Each bearing in XJTU-SY has two sensor channels:
- Assay 1 → horizontal accelerometer
- Assay 2 → vertical accelerometer

In [3]:
# Define the group once — reuse across any study
# Assay 1 = horizontal accelerometer, Assay 2 = vertical accelerometer
both_channels = AssayGroup([1, 2], name="Horizontal + Vertical")

print(both_channels)

AssayGroup(name='Horizontal + Vertical', assay_ids=[1, 2])


## 2. Lifecycle features for both channels

Pass the group directly to `study.lifecycle_features(group=...)` — no intermediate binding object to name.  
Returns `dict[label, DataFrame]` — one entry per assay in the group.


In [ ]:
lc_both = study.lifecycle_features(both_channels, file_type="raw", n_workers=4)

for label, df in lc_both.items():
    print(f"{label}: {len(df)} runs, columns = {list(df.columns)}")


Bearing 1_1 — Accel_Axial: 123 runs, columns = ['run_id', 'run_number', 'study_id', 'assay_id', 'rms', 'max', 'mean', 'peak2peak', 'kurtosis', 'std', 'crest_factor', 'skewness', 'fv_Fault Type', 'fv_Bearing Lifetime', 'fv_Motor speed', 'fv_Pressure Axial', 'fv_Pressure Radial']
Bearing 1_1 — Accel_Radial: 123 runs, columns = ['run_id', 'run_number', 'study_id', 'assay_id', 'rms', 'max', 'mean', 'peak2peak', 'kurtosis', 'std', 'crest_factor', 'skewness', 'fv_Fault Type', 'fv_Bearing Lifetime', 'fv_Motor speed', 'fv_Pressure Axial', 'fv_Pressure Radial']


## 3. Plot the grouped channels side by side

Pass the dict directly to `plot_multi_lifecycle()` to overlay both channels.

In [5]:
fig = plotter.plot_multi_lifecycle(
    lc_both,
    feature="rms",
    title="Bearing 1_1 — Horizontal vs Vertical RMS Degradation",
)
bokeh_show(fig)

In [6]:
fig = plotter.plot_multi_lifecycle(
    lc_both,
    feature="kurtosis",
    title="Bearing 1_1 — Horizontal vs Vertical Kurtosis",
)
bokeh_show(fig)

## 4. Cross-study comparison with a group

Call `compare_with()` directly on the **descriptor** (`both_channels`) — no study binding needed.  
All studies are explicit: what you pass is exactly what you get.


In [ ]:
study_11 = wrapper.study("Bearing 1_1")
study_12 = wrapper.study("Bearing 1_2")
study_13 = wrapper.study("Bearing 1_3")

# Call compare_with on the descriptor — no prior study.assay_group() needed
cross = both_channels.compare_with(
    [study_11, study_12, study_13],
    file_type="raw",
    n_workers=4,
)

print(f"{len(cross)} traces loaded:")
for key in cross:
    print(f"  {key}")


6 traces loaded:
  Bearing 1_1 — Accel_Axial
  Bearing 1_1 — Accel_Radial
  Bearing 1_2 — Accel_Axial
  Bearing 1_2 — Accel_Radial
  Bearing 1_3 — Accel_Axial
  Bearing 1_3 — Accel_Radial


In [8]:
fig = plotter.plot_multi_lifecycle(
    cross,
    feature="rms",
    title="35 Hz / 12 kN — Both Channels, Three Bearings",
)
bokeh_show(fig)

## 5. Use compare_studies with an AssayGroup

`wrapper.compare_studies()` also accepts an `assay_group` argument — this is the shortest form for large comparisons.

Only the studies you pass are included.

In [10]:
lc_all = wrapper.compare_studies(
    ["Bearing 2_1", "Bearing 2_3", "Bearing 2_5"],
    assay_group=both_channels,
    file_type="raw",
    n_workers=4,
)

fig = plotter.plot_multi_lifecycle(
    lc_all,
    feature="rms",
    title="compare_studies() with AssayGroup — 35 Hz / 12 kN",
)
bokeh_show(fig)